# 01 · Exploración del Chilean Waiting List Corpus (CWLC)

### Para qué sirve este notebook

Antes de comparar modelos hay que **conocer bien los datos**. Este notebook responde cuatro preguntas:

1. ¿Cuántas oraciones y entidades hay, y de qué tipos?
2. ¿Cómo se ve el texto real? ¿Es parecido al español "de manual"?
3. ¿Qué tan frecuentes son las abreviaturas y cuáles son?
4. ¿Cuántas enfermedades están escritas de forma abreviada? Esto es clave para la hipótesis H3 del proyecto.

**Sobre el corpus:** el CWLC reúne derivaciones (interconsultas) de las listas de espera de hospitales públicos chilenos, escritas por médicos generales de atención primaria. Están anonimizadas y fueron anotadas por médicos y dentistas (Báez et al., 2020 y 2022). Usamos la versión de 5.000 derivaciones publicada por el grupo PLN@CMM de la Universidad de Chile.

In [ ]:
from pathlib import Path
import sys

# Raíz del proyecto: funciona si abres el notebook desde VS Code en la carpeta notebooks/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
print("Raíz del proyecto:", RAIZ)

import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from datos.cwlc import TIPOS, cargar_split, resumen, es_abreviada

carpeta = RAIZ / "data" / "raw" / "cwlc"
datos = {s: cargar_split(carpeta, s) for s in ["train", "dev", "test"]}
print({s: len(o) for s, o in datos.items()})

## 1. Tamaño del corpus y tipos de entidad

Cada entidad es un trozo de texto que un médico marcó con un tipo. Los 7 tipos del corpus:

| Tipo | Significado | Ejemplo |
|---|---|---|
| Disease | Enfermedad o diagnóstico | "DM2", "caries" |
| Finding | Hallazgo clínico, signo o síntoma | "dolor", "aumento de volumen" |
| Abbreviation | Abreviatura o sigla | "HTA", "tto", "pcte" |
| Body_Part | Parte del cuerpo | "rodilla derecha", "pieza 3.6" |
| Procedure | Procedimiento | "exodoncia", "ecografía" |
| Medication | Fármaco | "metformina", "losartán" |
| Family_Member | Familiar | "madre", "hermano" |

In [ ]:
tabla = pd.DataFrame({s: resumen(o) for s, o in datos.items()}).T
tabla

In [ ]:
conteo = tabla.loc["train", [f"n_{t}" for t in TIPOS]].sort_values()
conteo.index = [i.replace("n_", "") for i in conteo.index]
ax = conteo.plot.barh(figsize=(7, 3.5), color="#4C72B0")
ax.set_xlabel("Número de entidades (train)")
ax.set_title("Entidades por tipo en el CWLC")
plt.tight_layout(); plt.show()

**Cómo leerlo:** Disease, Finding y Abbreviation son los tipos más frecuentes, con más de 8.000 cada uno en entrenamiento. Medication y Family_Member son escasos. Eso importa al evaluar: con pocos ejemplos de un tipo, su F1 será menos estable (intervalos de confianza más anchos). Por eso empezamos con **Disease, Medication y Abbreviation** (`configs/experimentos.yaml`).

## 2. ¿Cómo se ve el texto real?

Veamos algunas oraciones con sus entidades marcadas entre corchetes. Fíjense en las mayúsculas, las abreviaturas y el estilo telegráfico: es muy distinto de un caso clínico publicado en una revista.

In [ ]:
def marcar(oracion, tipos=("Disease", "Abbreviation", "Medication")):
    """Devuelve el texto con [mención|Tipo] para los tipos pedidos (sin anidamiento visual)."""
    texto, salida, pos = oracion.texto, [], 0
    ents = sorted([e for e in oracion.entidades if e[3] in tipos], key=lambda e: (e[1], -e[2]))
    for _, a, b, t in ents:
        if a < pos:
            continue  # entidad anidada dentro de otra ya marcada
        salida += [texto[pos:a], f"[{texto[a:b]}|{t}]"]
        pos = b
    return "".join(salida + [texto[pos:]])

ejemplos = [o for o in datos["train"] if any(e[3] == "Disease" for e in o.entidades) and 8 < len(o.tokens) < 30]
for o in ejemplos[:8]:
    print("•", marcar(o), "\n")

## 3. Las abreviaturas

El corpus marca cada abreviatura como entidad propia. Eso nos da, sin anotar nada, un **diccionario real** de cómo se abrevia en la APS chilena.

In [ ]:
abrev = Counter(o.mencion(e).lower() for o in datos["train"] for e in o.entidades if e[3] == "Abbreviation")
print("Abreviaturas distintas:", len(abrev), "| apariciones:", sum(abrev.values()))
pd.DataFrame(abrev.most_common(25), columns=["abreviatura", "veces"]).T

**Cómo leerlo:** aparecen siglas clínicas (HTA, DM2, IC, TAC), abreviaturas administrativas (APS, OBS, ANTEC) y palabras recortadas (tto, pcte, pza, izq). Algunas son **ambiguas**: "IC" puede ser interconsulta o insuficiencia cardíaca. Este tipo de texto es el que, según la literatura, más les cuesta a los modelos entrenados con español "de manual".

## 4. ¿Cuántas enfermedades están abreviadas?

Una enfermedad cuenta como **abreviada** si se superpone con una abreviatura anotada (por ejemplo "DM2", "HTA"). Así podemos comparar, en los resultados, el F1 en enfermedades abreviadas frente a las escritas completas. Ningún estudio publicado lo ha medido.

In [ ]:
filas = []
for s, oraciones in datos.items():
    enf = [(e, o) for o in oraciones for e in o.entidades if e[3] == "Disease"]
    n_ab = sum(es_abreviada(e, o) for e, o in enf)
    filas.append({"split": s, "enfermedades": len(enf), "abreviadas": n_ab, "% abreviadas": round(100 * n_ab / len(enf), 1)})
pd.DataFrame(filas).set_index("split")

In [ ]:
ab = [o.mencion(e) for o in datos["train"] for e in o.entidades if e[3] == "Disease" and es_abreviada(e, o)]
print("Ejemplos de enfermedades abreviadas:", Counter(m.upper() for m in ab).most_common(15))

**Implicancia para el diseño:** en el test hay alrededor de un centenar de enfermedades abreviadas, una de cada diez. Alcanza para medir la diferencia, pero los intervalos de confianza de ese grupo serán más anchos, y así lo reportaremos.

## 5. Largo de las oraciones

El largo importa para los LLM (cuánto texto leen por consulta) y para los BERT (tienen un límite de tokens).

In [ ]:
largos = pd.Series([len(o.tokens) for o in datos["train"]])
print(largos.describe().round(1))
ax = largos.clip(upper=80).plot.hist(bins=40, figsize=(7, 3), color="#55A868")
ax.set_xlabel("Palabras por oración (recortado en 80)"); ax.set_title("Largo de las oraciones del CWLC (train)")
plt.tight_layout(); plt.show()

## Conclusiones de la exploración

- El corpus es grande para ser español clínico: más de 35.000 entidades en entrenamiento, con splits fijos que permiten compararnos con Báez et al. (2022).
- El texto es real y "sucio": mayúsculas, abreviaturas, estilo telegráfico y siglas propias del sistema chileno.
- Existe un diccionario empírico de abreviaturas, y una de cada diez enfermedades está escrita abreviada. Con eso se puede evaluar H3 sin anotar nada.
- Las oraciones son cortas, lo que facilita usar LLM con varios ejemplos en el prompt (few-shot).

**Siguiente notebook:** `02_e1_encoder_ajustado.ipynb`, la línea base con un BERT clínico.